# Text-only SFT

Train on [JOSIE instructions](https://huggingface.co/datasets/mlx-community/JOSIE-v2-Instruct-5K)
with Qwen3.5-2B. Learn from assistant responses using the model's chat template.


## Setup

Use an Apple Silicon Mac and the Python environment selected by your notebook kernel.
Install once from your checkout, then restart the kernel:

```sh
python -m pip install -e "/absolute/path/to/mlx-vlm[train]" jupyterlab ipykernel
```

Run the cells in order. The first load downloads the model and dataset.


In [ ]:
from pathlib import Path

import mlx.core as mx
import pandas as pd
from datasets import load_dataset

from mlx_vlm import load
from mlx_vlm.prompt_utils import apply_chat_template
from mlx_vlm.trainer import Trainer, VLMTrainingArgs, prepare_model_for_training

## 1. Settings

Edit the model, data, and output path below. `iters` counts batches;
four accumulation steps make one optimizer update. Paths use the kernel's working directory.

Set sample limits to `None` to use the entire partition.


In [ ]:
MODEL_ID = "mlx-community/Qwen3.5-2B-4bit"
DATASET_ID = "mlx-community/JOSIE-v2-Instruct-5K"
VALIDATION_FRACTION = 0.05
MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES = None
OUTPUT_DIR = Path("adapters/vlm/sft_text_only")
SEED = 42

# Adapters
TRAIN_TYPE = "lora"  # "lora", "dora", or "full"
CHECKPOINT_PATH = None
LORA_RANK = 8
LORA_ALPHA = 16

# Training
args = VLMTrainingArgs(
    iters=100,
    batch_size=1,
    learning_rate=1e-4,
    gradient_accumulation_steps=4,
    max_seq_length=2048,
    train_on_completions=True,
    cache_size=8,
    grad_checkpoint=True,
    compile=False,
    steps_per_report=10,
    steps_per_eval=50,
    steps_per_save=100,
    adapter_file=str(OUTPUT_DIR / "weights.safetensors"),
)
mx.random.seed(SEED)
args.validate()

## 2. Data

JOSIE already contains `messages`. Remove conversations without a non-empty final
assistant response, then create a seeded training/validation split.


In [ ]:
dataset = load_dataset(DATASET_ID, split="train")
if "messages" not in dataset.column_names:
    raise ValueError(f"Expected a messages column, got {dataset.column_names}.")


def has_assistant_target(row):
    messages = row["messages"]
    return bool(
        messages
        and messages[-1].get("role") == "assistant"
        and isinstance(messages[-1].get("content"), str)
        and messages[-1]["content"].strip()
    )


source_rows = len(dataset)
dataset = dataset.filter(
    has_assistant_target, desc="Keep conversations with assistant targets"
)
dropped_rows = source_rows - len(dataset)
print({"source_rows": source_rows, "dropped_without_assistant_target": dropped_rows})
if len(dataset) < 2:
    raise ValueError("Need at least two conversations with assistant targets to split.")

partitions = dataset.train_test_split(test_size=VALIDATION_FRACTION, seed=SEED)


def limit_rows(rows, maximum):
    if maximum is None:
        return rows
    if maximum < 1:
        raise ValueError("Sample limits must be positive or None.")
    return rows.select(range(min(maximum, len(rows))))


train_rows = limit_rows(partitions["train"], MAX_TRAIN_SAMPLES)
eval_rows = limit_rows(partitions["test"], MAX_EVAL_SAMPLES)

assert len(train_rows) >= args.batch_size, "Reduce batch_size or add training rows."
assert len(eval_rows) >= args.batch_size, "Validation needs a complete batch."
print({"train_rows": len(train_rows), "validation_rows": len(eval_rows)})
print(train_rows.features)

## 3. Model

Preview one conversation with the chat template. Pass the original messages to
the trainer so it can mask the prompt correctly.


In [ ]:
model, processor = load(MODEL_ID)
formatted_sample = apply_chat_template(
    processor,
    model.config,
    train_rows[0]["messages"],
    num_images=0,
    add_generation_prompt=False,
)
print(formatted_sample[:2000])

### Prepare adapters

LoRA trains the language adapters while freezing the base. `TRAIN_TYPE` also accepts
`"dora"` or `"full"`; full training unfreezes all weights and needs more memory.
Set `CHECKPOINT_PATH` to saved weights to continue with a fresh optimizer.
See the [training guide](../../docs/training.md) for quantization and target-layer options.


In [ ]:
model = prepare_model_for_training(
    model,
    train_type=TRAIN_TYPE,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    checkpoint_path=CHECKPOINT_PATH,
)

## 4. Train

Only the final assistant response contributes to the loss.


In [ ]:
trainer = Trainer(
    model,
    args,
    task="vlm",
    algorithm="sft",
    processing_class=processor,
    train_dataset=train_rows,
    eval_dataset=eval_rows,
    seed=SEED,
)

In [ ]:
metrics = trainer.train()

### Metrics

The table shows the last five training reports. All metrics remain in `metrics`
and `trainer.log_history`, including loss diagnostics, timing, and memory.
`trained_tokens` counts supervised targets; `total_vision_tokens` counts expanded
image/video input positions. Processing token/s includes forward, backward, and updates.


In [ ]:
report_history = pd.DataFrame(trainer.log_history)
columns = {
    "iteration": "step",
    "train_loss": "loss",
    "learning_rate": "lr",
    "trained_tokens": "trained tokens",
    "total_vision_tokens": "vision tokens",
    "processed_tokens_per_second": "processed tok/s",
    "peak_memory": "peak GB",
}
recent_reports = (
    report_history.dropna(subset=["train_loss"])
    .tail(5)
    .reindex(columns=columns)
    .rename(columns=columns)
    .set_index("step")
)
recent_reports.round({"loss": 4, "processed tok/s": 1, "peak GB": 2})

## 5. Evaluate

Check loss on held-out conversations. `max_seq_length` can truncate long responses;
review generated answers as well when assessing instruction following.


In [ ]:
assert mx.isfinite(mx.array(metrics["train_loss"])).item(), "Non-finite training loss."
validation = trainer.evaluate() if eval_rows is not None else {}
pd.DataFrame([validation]).filter(
    items=["val_loss", "val_token_accuracy", "val_perplexity", "val_num_vision_tokens"]
).round(4)

## 6. Save

Keep `weights.safetensors` and `adapter_config.json` together. The loop also saves
the final checkpoint automatically.


In [ ]:
trainer.save_adapter()
print(Path(args.adapter_file).resolve())

### Reload later

Load the original base model, then attach the saved weights:

```python
model, processor = load(MODEL_ID)
model = prepare_model_for_training(
    model, train_type=TRAIN_TYPE, checkpoint_path=args.adapter_file,
)
model.eval()
```

More options: [training guide](../../docs/training.md).
